# Estimating Equilibrium Climate Sensitivity (ECS) in CMIP6 models

**Adapted from** https://github.com/pangeo-gallery/cmip6/blob/master/ECS_Gregory_method.ipynb

---

In this module, we'll be looking at Equilibrium Climate Sensitivity as defined in our text. We'll be using a method called the "Gregory Method" to approximate the ECS of CMIP6 models.

### Python packages

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe
import cartopy
import dask
from tqdm.autonotebook import tqdm  # Fancy progress bars for our loops!
import intake
import fsspec

%matplotlib inline
plt.rcParams['figure.figsize'] = 12, 6
%config InlineBackend.figure_format = 'retina' 

Now that you know how to download CMIP6 data from the last module, we'll provide data here. It's important to know in your own research that downloading data can often be a time sink so its good to plan ahead.

In [ ]:
big_ds = xr.open_dataset('abrupt-4xCO2_global_annual_means.nc')

# THIS IS WHERE THIS STARTS

I'll provide this data for ease of access

### Calculated Derived Variables

We need to calculate the net radiative imbalance, plus the anomaly of the abrupt 4xCO2 run compared to the piControl run.

In [ ]:
big_ds['imbalance'] = big_ds['rsdt'] - big_ds['rsut'] - big_ds['rlut']

ds_mean = big_ds[['tas', 'imbalance']].sel(experiment_id='piControl').mean(dim='year')
ds_anom = big_ds[['tas', 'imbalance']] - ds_mean

# add some metadata
ds_anom.tas.attrs['long_name'] = 'Global Mean Surface Temp Anom'
ds_anom.tas.attrs['units'] = 'K'
ds_anom.imbalance.attrs['long_name'] = 'Global Mean Radiative Imbalance'
ds_anom.imbalance.attrs['units'] = 'W m$^{-2}$'

ds_anom

### Plot Timeseries

Here we plot the global mean surface temperature for each model:

In [ ]:
ds_anom.tas.plot.line(col='source_id', x='year', col_wrap=5)

We can see that the models cover different time intervals. Let's limit the rest of our analysis to the first 150 years.

In [ ]:
# limit to the gregory 150-year period
first_150_years = slice(0, 149)
ds_anom.tas.sel(year=first_150_years).plot.line(col='source_id', x='year', col_wrap=5)

Same thing for radiative imbalance.

In [ ]:
ds_anom.imbalance.sel(year=first_150_years).plot.line(col='source_id', x='year', col_wrap=5)

### Calculate ECS

In [ ]:
ds_abrupt = ds_anom.sel(year=first_150_years, experiment_id='abrupt-4xCO2').reset_coords(drop=True)

def calc_ecs(ds):
    # Some sources don't have all 150 years, drop those missing values.
    a, b = np.polyfit(ds.tas.dropna("year"),
                      ds.imbalance.dropna("year"), 1)
    ecs = -0.5 * (b/a)
    return xr.DataArray(ecs)

ds_abrupt['ecs'] = ds_abrupt.groupby('source_id').apply(calc_ecs)
ds_abrupt

Reproduce a plot similar to [Mark Zelinka's](https://twitter.com/mzelinka/status/1255534531144085513):

In [ ]:
fg = ds_abrupt.plot.scatter(x='tas', y='imbalance', col='source_id', col_wrap=5)

def calc_and_plot_ecs(x, y, **kwargs):
    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]
    a, b = np.polyfit(x, y, 1)
    ecs = -0.5 * b/a
    plt.autoscale(False)
    plt.plot([0, 10], np.polyval([a, b], [0, 10]), 'k')
    plt.text(4, 6, f'ECS = {ecs:3.2f}', fontdict={'weight': 'bold', 'size': 12})
    plt.grid()
    
fg.map(calc_and_plot_ecs, 'tas', 'imbalance')

In [ ]:
ds_abrupt.ecs.plot.hist();

In [ ]:
ds_abrupt.ecs.to_dataframe().sort_values('ecs').plot(kind='bar')